In [ ]:
!pip install speechbrain torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.1/782.1 kB 45.0 MB/s eta 0:00:00


In [ ]:
import os
import torch
import torchaudio
import urllib.request
from speechbrain.inference.speaker import EncoderClassifier, SpeakerRecognition

In [ ]:
#we will first download the test audio files s1,s2,s3 s1 and s2 are teh same speaker saying different phrases

base_url = "https://raw.githubusercontent.com/speechbrain/speechbrain/develop/tests/samples/ASR/"
files = ["spk1_snt1.wav", "spk1_snt2.wav", "spk2_snt1.wav"]

for f in files:
    if not os.path.exists(f):
        urllib.request.urlretrieve(base_url + f, f)
        print(f"Downloaded {f}")

Downloaded spk1_snt1.wav
Downloaded spk1_snt2.wav
Downloaded spk2_snt1.wav


In [ ]:
# we will use the model from speech brains to embed the audio files in to embeddings
# (it isolates the words and focuses on the physiological characteristics)
# to output dense 192 dimensional vector

print("\n--- ENROLLMENT PHASE ---")
# Load the pretrained ECAPA-TDNN model for extracting embeddings
classifier = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="pretrained_models/spkrec-ecapa-voxceleb"
)

# Load the target user's audio
signal, fs = torchaudio.load("spk1_snt1.wav")

# Extract the embedding (a mathematical representation of the voice)
embeddings = classifier.encode_batch(signal)
print(f"Embedding extracted! Shape: {embeddings.shape}")

# Store the embedding to disk (Simulating saving to a database)
torch.save(embeddings, "speaker1_enrollment.pt")
print("Saved embedding to 'speaker1_enrollment.pt'")

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached



--- ENROLLMENT PHASE ---


hyperparams.yaml:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


embedding_model.ckpt: reconstructing file:   0%|          |  0.00B / 83.3MB            

...

classifier.ckpt: downloading bytes:           |  0.00B            

INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


label_encoder.txt:   0%|          | 0.00/129k [00:00<?, ?B/s]

INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder


Embedding extracted! Shape: torch.Size([1, 1, 192])
Saved embedding to 'speaker1_enrollment.pt'


In [ ]:
# embedding the audio file calculate the cosine similarity and compare against a perset threshold
# by default the threshold value is 0.25

print("\n--- VERIFICATION PHASE ---")
# Load the verification module
verification = SpeakerRecognition.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="pretrained_models/spkrec-ecapa-voxceleb"
)

# Test A: Verify against itself (Same Speaker, Different Phrase)
score_same, prediction_same = verification.verify_files("spk1_snt1.wav", "spk1_snt2.wav")
print(f"Test 1 (Same Speaker): Match? {prediction_same.item()} | Score: {score_same.item():.4f}")

# Test B: Verify against someone else (Different Speaker)
score_diff, prediction_diff = verification.verify_files("spk1_snt1.wav", "spk2_snt1.wav")
print(f"Test 2 (Different Speaker): Match? {prediction_diff.item()} | Score: {score_diff.item():.4f}")

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Using symlink found at '/content/pretrained_models/spkrec-ecapa-voxceleb/hyperparams.yaml'



--- VERIFICATION PHASE ---


INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Using symlink found at '/content/pretrained_models/spkrec-ecapa-voxceleb/embedding_model.ckpt'
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Using symlink found at '/content/pretrained_models/spkrec-ecapa-voxceleb/mean_var_norm_emb.ckpt'
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Using symlink found at '/content/pretrained_models/spkrec-ecapa-voxceleb/classifier.ckpt'
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Using symlink found at '/content/pretrained_models/spkrec-ecapa-voxceleb/label_encoder.ckpt'
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder


Test 1 (Same Speaker): Match? True | Score: 0.7295
Test 2 (Different Speaker): Match? False | Score: 0.0762


In [ ]:
print()